In [1]:
import pickle
import json

## Load Data

In [2]:
with open("artificial_profiles_scores.pkl", "rb") as f:
    interactions = pickle.load(f)

with open("artificial_profiles.json", "r", encoding="utf-8") as f:
    features_by_user = json.load(f)

with open("titles_with_tags_dict.pkl", "rb") as f:
    features_by_item = pickle.load(f)

print("Data loaded successfully:")
print(f"  - Users: {len(interactions)}")
print(f"  - Items: {len(features_by_item)}")
print(
    f"  - Total interactions: {sum(len(ratings) for ratings in interactions.values())}"
)

Data loaded successfully:
  - Users: 31
  - Items: 1164
  - Total interactions: 359


In [3]:
sample_user = list(features_by_user.keys())[0]
print(f"Sample user: {sample_user}")
print(f"Bio: {features_by_user[sample_user]['bio'][:200]}...")
print(f"Tags: {features_by_user[sample_user]['tags']}")

Sample user: global_economics_and_geopolitics_analyst
Bio: A highly analytical researcher deeply engaged in monitoring global economic trends, geopolitical risks, international trade, and the socio-economic development of various regions (especially BRICS, As...
Tags: ['global_economy', 'geopolitics', 'macroeconomics', 'international_economics', 'data_analysis', 'forex', 'regional_development', 'brics', 'political_economy', 'international_relations', 'data_monitoring', 'time_series', 'system_thinking', 'geopolitics_of_BRICS', 'investment_analysis', 'economic_forecasting', 'trade_policy', 'market_forecasting', 'data_mining', 'quantitative_finance', 'global_systems', 'regional_analysis', 'economic_development']


In [4]:
sample_item = list(features_by_item.keys())[0]
print(f"Sample item: {sample_item[:80]}...")
print(f"Tags: {features_by_item[sample_item]}")

Sample item: Исследование приоритетов и механизмов реализации отраслевых (секторальных) полит...
Tags: ['international_relations', 'political_economics', 'policy_analysis', 'BRICS', 'geopolitics', 'international_policy', 'comparative_politics']


In [5]:
from lightfm.data import Dataset
from lightfm import LightFM

/Users/antonshishkov/Projects/diploma/.venv/lib/python3.14/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [6]:
dataset = Dataset()

In [7]:
import numpy as np
import pandas as pd

In [8]:
data = []
for user in interactions:
    for payload in interactions[user]:
        data.append(
            (user, json.loads(payload).get("title"), interactions[user][payload])
        )

df = pd.DataFrame(data=data, columns=["user_name", "item_title", "rating"])

In [9]:
dataset.fit(df["user_name"].unique(), df["item_title"].unique())

In [10]:
user_features = set()

for user in features_by_user:
    user_features = user_features.union(features_by_user[user]["tags"])

user_features = np.array(list(user_features))

In [11]:
dataset.fit_partial(user_features=user_features)

In [12]:
item_features = set()

for title in features_by_item:
    item_features = item_features.union(features_by_item[title])

item_features = np.array(list(item_features))

In [13]:
dataset.fit_partial(item_features=item_features)

In [14]:
num_users, num_items = dataset.interactions_shape()
num_users, num_items

(31, 316)

In [15]:
lightfm_mapping = dataset.mapping()
lightfm_mapping = {
    "users_mapping": lightfm_mapping[0],
    "user_features_mapping": lightfm_mapping[1],
    "items_mapping": lightfm_mapping[2],
    "item_features_mapping": lightfm_mapping[3],
}
print("users_mapping len - ", len(lightfm_mapping["users_mapping"]))
print("user_features_mapping len - ", len(lightfm_mapping["user_features_mapping"]))
print("items_mapping len - ", len(lightfm_mapping["items_mapping"]))
print(
    "Users item_features_mapping len - ", len(lightfm_mapping["item_features_mapping"])
)

users_mapping len -  31
user_features_mapping len -  229
items_mapping len -  316
Users item_features_mapping len -  2177


In [16]:
lightfm_mapping["users_inv_mapping"] = {
    v: k for k, v in lightfm_mapping["users_mapping"].items()
}
lightfm_mapping["items_inv_mapping"] = {
    v: k for k, v in lightfm_mapping["items_mapping"].items()
}

In [17]:
num_user_features = dataset.user_features_shape()
num_show_features = dataset.item_features_shape()
print(
    "num user features: {} -> {}\nnum item features: {} -> {}.".format(
        num_user_features[1] - num_users,
        num_user_features[1],
        num_show_features[1] - num_items,
        num_show_features[1],
    )
)

num user features: 198 -> 229
num item features: 1861 -> 2177.


In [18]:
def df_to_tuple_iterator(df):
    return zip(*df.values.T)


def concat_last_to_list(t):
    return (t[0], list(t[1:])[0])


def df_to_tuple_list_iterator(df):
    return map(concat_last_to_list, zip(*df.values.T))

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
df["user_id"] = df.apply(
    lambda x: lightfm_mapping["users_mapping"][x["user_name"]], axis=1
)
df["item_id"] = df.apply(
    lambda x: lightfm_mapping["items_mapping"][x["item_title"]], axis=1
)

In [21]:
df.head(5)

,user_name,item_title,rating,user_id,item_id
0,global_economics_and_geopolitics_analyst,Группа поддержки профессионального самоопредел...,1,0,0
1,global_economics_and_geopolitics_analyst,Правовые аспекты обучения моделей искусственно...,3,0,1
2,global_economics_and_geopolitics_analyst,Разработка веб-сайта на конструкторе веб-сайто...,1,0,2
3,global_economics_and_geopolitics_analyst,Ценностные паттерны в консервативном сегменте ...,2,0,3
4,global_economics_and_geopolitics_analyst,Формирование и развитие маркетинговой стратеги...,2,0,4


In [22]:
train, test = train_test_split(df, test_size=0.2, random_state=24)

In [23]:
train_mat, train_mat_weights = dataset.build_interactions(
    df_to_tuple_iterator(train[["user_name", "item_title"]])
)

In [24]:
data = []

for user in features_by_user:
    data.append((user, features_by_user[user]["tags"]))

df_users = pd.DataFrame(data=data, columns=["user_name", "features"])

In [25]:
df_users.head()

,user_name,features
0,global_economics_and_geopolitics_analyst,"[global_economy, geopolitics, macroeconomics, ..."
1,international_relations_and_geopolitics_expert,"[international_politics, geopolitics, regional..."
2,multilingual_linguistics_researcher,"[linguistics, translation_technology, corpus_p..."
3,education_and_cultural_development_expert,"[language_teaching, educational_materials, cul..."
4,sociocultural_anthropologist_and_politician,"[social_anthropology, political_analysis, cult..."


In [26]:
known_users_filter = df_users["user_name"].isin(df["user_name"].unique())
train_user_features = dataset.build_user_features(
    df_to_tuple_list_iterator(
        df_users.loc[known_users_filter, ["user_name", "features"]]
    )
)

In [27]:
data = []

for item in features_by_item:
    data.append((item, features_by_item[item]))

df_items = pd.DataFrame(data=data, columns=["item_title", "features"])

In [28]:
df_items.head()

,item_title,features
0,Исследование приоритетов и механизмов реализац...,"[international_relations, political_economics,..."
1,Антрополе - научно-популярный видео-подкаст о ...,"[anthropology, social_phenomena, media, podcas..."
2,"Разработка, создание и ведение сайта, посвящен...","[web_development, history, cultural_studies, d..."
3,Перевод с английского языка коллективной моног...,"[criminology, literature_review, translation, ..."
4,Сеть военно-политических союзов в Евразии: баз...,"[geopolitics, international_relations, databas..."


In [29]:
known_items_filter = df_items["item_title"].isin(df["item_title"].unique())
train_items_features = dataset.build_item_features(
    df_to_tuple_list_iterator(
        df_items.loc[known_items_filter, ["item_title", "features"]]
    )
)

In [30]:
lfm_model = LightFM(
    no_components=64, learning_rate=0.05, loss="warp", max_sampled=5, random_state=42
)

In [31]:
import tqdm

In [32]:
num_epochs = 15
for _ in tqdm.tqdm(range(num_epochs), total=num_epochs):
    lfm_model.fit_partial(
        train_mat,
        user_features=train_user_features,
        item_features=train_items_features,
        num_threads=4,
    )

100%|██████████| 15/15 [00:00<00:00, 553.67it/s]


In [33]:
sorted(test["user_name"].unique())

['ai_and_nlp_developer',
 'ai_policy_and_social_impact_analyst',
 'cultural_humanities_researcher_media_studies',
 'data_driven_policy_analyst',
 'digital_marketing_and_media_strategy',
 'digital_platform_developer_and_educator',
 'education_and_cultural_development_expert',
 'educational_psychologist_and_methodologist',
 'educational_tech_developer',
 'financial_economist_and_analyst',
 'geopolitics_and_international_relations_expert',
 'global_economics_and_geopolitics_analyst',
 'historical_culture_researcher_media_specialist',
 'international_relations_and_geopolitics_expert',
 'legal_policy_and_ethics_researcher',
 'literary_and_cultural_analyst_cross_cultural',
 'media_and_communication_strategist',
 'media_and_communications_specialist',
 'media_and_culture_strategist',
 'media_journalism_and_cultural_studies_expert',
 'multidisciplinary_tech_and_language_developer',
 'multilingual_linguistics_researcher',
 'multilingual_nlp_and_ai_specialist',
 'psychological_researcher_and_wel

In [34]:
top_N = 10
user_name = "ai_and_nlp_developer"
row_id = lightfm_mapping["users_mapping"][user_name]
print(f"Рекомендации для пользователя {user_name}, номер строки - {row_id}")

Рекомендации для пользователя ai_and_nlp_developer, номер строки - 20


In [35]:
all_cols = list(lightfm_mapping["items_mapping"].values())
len(all_cols)

316

In [36]:
pred = lfm_model.predict(
    row_id,
    all_cols,
    user_features=train_user_features,
    item_features=train_items_features,
    num_threads=4,
)

In [37]:
top_cols = np.argpartition(pred, -np.arange(top_N))[-top_N:][::-1]

In [38]:
recs = pd.DataFrame({"col_id": top_cols})
recs["item_title"] = recs["col_id"].map(lightfm_mapping["items_inv_mapping"].get)

In [39]:
recs

,col_id,item_title
0,187,Театральный киноклуб
1,146,Просветительский проект «Голос Востока»
2,240,Разработка системы учёта доходов и расходов в ...
3,184,Тестовая ВКР по физике
4,39,Создание мобильного приложения для изучения РЖ...
5,276,"""Портфельная аналитика российских активов"""
6,214,Администрирование консультативной деятельности...
7,107,Сравнительный анализ концепций работы со спонс...
8,139,Автоматизация регулярного отчета для отдела e-...
9,33,Преимущества низковолатильных ETF с точки зрен...
